# 5. Downstream capabilities on the shared latent Z

```
                  Encoder
                     |
                     v
              Robust latent Z
                     |
    +--------+-------+-------+--------+
    |        |       |       |        |
Reconstruct Contrast Classify Retrieve Anomaly
```

The same pooled latent Z that the decoder cross-attends into for reconstruction also feeds four other heads (`model/downstream.py`): contrastive learning, classification, retrieval, and anomaly detection. This notebook trains the base autoencoder on `data/sample_corpus.tsv` (120 sentences labeled across 6 topics) and exercises every head, plotting a graph for each. It imports the same `phase_*` functions that `run_downstream_demo.py` uses from the command line, so the notebook and the script can't drift apart.

Same honesty caveat as the rest of this repo: 120 sentences and a ~45M-param model are demonstration scale, not benchmark scale.

In [ ]:
import sys, pathlib, random
sys.path.append(str(pathlib.Path.cwd().parent))

import torch
import matplotlib.pyplot as plt

from run_downstream_demo import (
    load_labeled_corpus, build_model_and_tokenizer, encode_to_latent, pca_2d,
    phase_reconstruction, phase_contrastive, phase_classification, phase_retrieval, phase_anomaly,
)

## Setup: corpus, tokenizer, base model

In [ ]:
CORPUS_PATH = "../data/sample_corpus.tsv"
TOKENIZER_DIR = "../tokenizer/vocab"
SEQ_LEN = 64
BATCH_SIZE = 8

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")

labels, texts = load_labeled_corpus(CORPUS_PATH)
categories = sorted(set(labels))
print(f"{len(texts)} sentences, {len(categories)} categories: {categories}")

plain_corpus_path = pathlib.Path("../data/sample_corpus.txt")
plain_corpus_path.write_text("\n".join(texts) + "\n", encoding="utf-8")

tok, cfg, model = build_model_and_tokenizer(str(plain_corpus_path), TOKENIZER_DIR, 4000, SEQ_LEN, device)
print(f"model parameters: {model.num_parameters():,}")

opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
rng = random.Random(0)

## Reconstruction (the base objective the latent Z is built on)

In [ ]:
RECON_STEPS = 1500
recon_losses = phase_reconstruction(model, tok, cfg, texts, opt, RECON_STEPS, BATCH_SIZE, SEQ_LEN, device, rng)

plt.figure(figsize=(8, 4.5))
plt.plot(recon_losses)
plt.xlabel("step")
plt.ylabel("cross-entropy loss")
plt.title(f"Reconstruction training ({model.num_parameters()/1e6:.1f}M params, {device})")
plt.show()

## Contrastive: two noise views of the same sentence should land close in Z

In [ ]:
same_pair_loss, shuffled_loss = phase_contrastive(model, tok, cfg, texts, SEQ_LEN, device, rng)
print(f"same-sentence pairs:  {same_pair_loss:.4f}")
print(f"shuffled/mismatched:  {shuffled_loss:.4f}")
print("(lower = more similar; same-sentence pairs should score lower)")

plt.figure(figsize=(5, 4.5))
bars = plt.bar(["same sentence\n(2 noise views)", "shuffled\n(mismatched)"],
               [same_pair_loss, shuffled_loss], color=["#4C72B0", "#C44E52"])
plt.ylabel("contrastive loss (lower = more similar)")
plt.title("Contrastive: matching vs. mismatched pairs")
for b, v in zip(bars, [same_pair_loss, shuffled_loss]):
    plt.text(b.get_x() + b.get_width() / 2, v, f"{v:.4f}", ha="center", va="bottom")
plt.show()

## Classification: a linear head on pooled Z predicting topic category

In [ ]:
CLASSIFIER_STEPS = 300
clf, clf_losses, clf_accs, full_acc = phase_classification(
    model, tok, cfg, texts, labels, categories, 1e-3, CLASSIFIER_STEPS, BATCH_SIZE, SEQ_LEN, device, rng
)
print(f"final accuracy over the full corpus: {full_acc:.2%}  (random baseline: {1/len(categories):.2%})")

fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax1.plot(clf_losses, color="#4C72B0", label="loss")
ax1.set_xlabel("step")
ax1.set_ylabel("cross-entropy loss", color="#4C72B0")
ax2 = ax1.twinx()
ax2.plot(clf_accs, color="#55A868", label="batch accuracy")
ax2.set_ylabel("batch accuracy", color="#55A868")
ax2.set_ylim(0, 1)
plt.title(f"Classification head training ({len(categories)}-way)")
plt.show()

## Retrieval: nearest neighbors in the latent space, and a PCA view of the whole embedding space

In [ ]:
corpus_pooled, top_idx, top_scores = phase_retrieval(model, tok, cfg, texts, labels, SEQ_LEN, device, query_idx=0, k=4)
print(f"query: {texts[0]}  [{labels[0]}]")
for i, score in zip(top_idx, top_scores):
    marker = " (self)" if i == 0 else ""
    print(f"  {score:.3f}  [{labels[i]}] {texts[i]}{marker}")

In [ ]:
coords = pca_2d(corpus_pooled)
plt.figure(figsize=(6.5, 6))
palette = plt.get_cmap("tab10")
for i, cat in enumerate(categories):
    idx = [j for j, l in enumerate(labels) if l == cat]
    plt.scatter(coords[idx, 0], coords[idx, 1], label=cat, color=palette(i), s=35, alpha=0.8)
plt.legend(fontsize=8)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Latent Z embedding space (PCA), colored by topic")
plt.show()

## Anomaly detection: reconstruction error as an anomaly score

In [ ]:
anomaly_scores = phase_anomaly(model, tok, cfg, texts, SEQ_LEN, device, rng)
for label, (text, score) in anomaly_scores.items():
    print(f"{label}: {score:.3f}   \"{text}\"")

plt.figure(figsize=(5, 4.5))
keys = list(anomaly_scores.keys())
values = [anomaly_scores[k][1] for k in keys]
bars = plt.bar(["in-distribution", "out-of-distribution"], values, color=["#4C72B0", "#C44E52"])
plt.ylabel("reconstruction error (anomaly score)")
plt.title("Anomaly detection via reconstruction error")
for b, v in zip(bars, values):
    plt.text(b.get_x() + b.get_width() / 2, v, f"{v:.2f}", ha="center", va="bottom")
plt.show()

## Summary

All five heads share one encoder and one latent Z: reconstruction trains it, and contrastive, classification, retrieval, and anomaly detection all read from it without needing a separate model. Scale the corpus and `d_model`/layer counts up (see `model/config.py` and the scaling table in `RAE1B_Colab_Complete.ipynb`) for anything beyond this demonstration.